# Tesis: Chatbot para Diagnóstico Vehicular Híbrido en Carabayllo
## Visualización de Entrenamiento de Modelos y Análisis Estadístico (Capítulo IV - Resultados)

Este cuaderno de Jupyter contiene la explicación paso a paso de:
1. **Entrenamiento del Modelo de Machine Learning** (TF-IDF + Random Forest) con el dataset de síntomas en español.
2. **Análisis de Métricas Comparativas** (Comparación de precisión, completitud de fichas y tiempos de diagnóstico).
3. **Contraste de Hipótesis mediante T-Student** para muestras relacionadas (Pág. 27 del PDF de la tesis).

## 1. Entrenamiento del Modelo de Machine Learning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import os

# 1. Cargar el dataset de síntomas mecánicos
df_sintomas = pd.read_csv('data/dataset_sintomas.csv', encoding='utf-8')
print(f'✓ Dataset cargado correctamente con {len(df_sintomas)} ejemplos.')
df_sintomas.head(10)

### Vectorización y Clasificación con Random Forest

In [ ]:
X = df_sintomas['sintoma']
y = df_sintomas['falla']

# Vectorizar texto con TF-IDF
vectorizador = TfidfVectorizer(lowercase=True, strip_accents='unicode')
X_vectorizado = vectorizador.fit_transform(X)

# Entrenar clasificador Random Forest
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_vectorizado, y)

predicciones = modelo.predict(X_vectorizado)
exactitud = accuracy_score(y, predicciones)
print(f'✓ Modelo entrenado con éxito. Exactitud del entrenamiento: {exactitud * 100:.2f}%')

### Matriz de Confusión
Visualicemos cómo se clasifican las categorías.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay.from_predictions(y, predicciones, ax=ax, xticks_rotation='vertical', cmap='Blues')
plt.title('Matriz de Confusión del Entrenamiento')
plt.show()

## 2. Análisis Estadístico y Comparativo (Pre-test vs Post-test)

In [ ]:
# Cargar los datos de los experimentos en el taller
df_tracker = pd.read_csv('data/tracker_diagnosticos.csv')

# Separar Pre-test y Post-test
pre_test = df_tracker[df_tracker['fase'] == 'Pre-test']
post_test = df_tracker[df_tracker['fase'] == 'Post-test']

print(f'Registros Pre-test: {len(pre_test)}')
print(f'Registros Post-test: {len(post_test)}')

### Comparación de Métricas de Rendimiento

In [ ]:
# 1. Precisión
precision_pre = (pre_test['prediccion_correcta'].sum() / len(pre_test)) * 100
precision_post = (post_test['prediccion_correcta'].sum() / len(post_test)) * 100

# 2. Completitud
completitud_pre = (pre_test['campos_completos'].sum() / len(pre_test)) * 100
completitud_post = (post_test['campos_completos'].sum() / len(post_test)) * 100

# 3. Tiempos Promedio
tiempo_prom_pre = pre_test['tiempo_diagnostico_minutos'].mean()
tiempo_prom_post = post_test['tiempo_diagnostico_minutos'].mean()

print('Fase Pre-test (Sin Chatbot):')
print(f'- Precisión de Fallas: {precision_pre:.2f}%')
print(f'- Completitud de Ficha: {completitud_pre:.2f}%')
print(f'- Tiempo de Diagnóstico Promedio: {tiempo_prom_pre:.2f} minutos')
print('-' * 50)
print('Fase Post-test (Con Chatbot):')
print(f'- Precisión de Fallas: {precision_post:.2f}%')
print(f'- Completitud de Ficha: {completitud_post:.2f}%')
print(f'- Tiempo de Diagnóstico Promedio: {tiempo_prom_post:.2f} minutos')

## 3. Contraste de Hipótesis (T-Student de muestras relacionadas)
Evaluamos la reducción de tiempos de diagnóstico (Pág. 27 del PDF de la tesis).

In [ ]:
from scipy import stats

t_stat, p_value = stats.ttest_rel(pre_test['tiempo_diagnostico_minutos'], post_test['tiempo_diagnostico_minutos'])

print(f'Estadístico T: {t_stat:.4f}')
print(f'P-Valor: {p_value:.8f}')

if p_value < 0.05:
    print('\n✓ CONCLUSIÓN CIENTÍFICA: Se rechaza la hipótesis nula (H0). El chatbot influye de manera significativa en la reducción de tiempos de diagnóstico en el taller.')
else:
    print('\nNo se rechaza H0. No hay diferencia estadísticamente significativa.')

### Gráficas de Resultados Comparativos

In [ ]:
# Configurar estilo de gráficos
sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 11})

# Gráfica de caja: Tiempos de Diagnóstico
plt.figure(figsize=(7, 5))
sns.boxplot(x='fase', y='tiempo_diagnostico_minutos', data=df_tracker, palette='Set2', width=0.5)
plt.title('Reducción de Tiempos de Diagnóstico (Minutos)')
plt.xlabel('Fase')
plt.ylabel('Minutos')
plt.show()

# Gráfica de barra: Comparativa de Calidad
metricas_data = {
    'Fase': ['Pre-test', 'Post-test', 'Pre-test', 'Post-test'],
    'Métrica': ['Precisión', 'Precisión', 'Completitud', 'Completitud'],
    'Porcentaje (%)': [precision_pre, precision_post, completitud_pre, completitud_post]
}
df_metricas = pd.DataFrame(metricas_data)

plt.figure(figsize=(8, 5))
sns.barplot(x='Métrica', y='Porcentaje (%)', hue='Fase', data=df_metricas, palette='Set1')
plt.title('Impacto en la Calidad del Diagnóstico Vehicular')
plt.ylabel('Porcentaje (%)')
plt.ylim(0, 115)
plt.show()